In [1]:
import os
import pickle
import torch
from tqdm import tqdm
from transformers import Qwen3VLForConditionalGeneration, AutoProcessor
from datasets import load_dataset, Dataset

save_base_dir = "path/to/calib"



# ==============================
# 1. Load dataset
# ==============================
print("Loading dataset...")
train_raw_dataset = Dataset.from_dict(
    load_dataset("AI4Math/MathVerse", 'testmini')['testmini'][:256]
)

# ==============================
# 2. Load model and processor
# ==============================
print("Loading model and processor...")
model = Qwen3VLForConditionalGeneration.from_pretrained(
    "Qwen/Qwen3-VL-2B-Instruct",
    torch_dtype="bfloat16",
    device_map="auto"
)
processor = AutoProcessor.from_pretrained("Qwen/Qwen3-VL-2B-Instruct")

# ==============================
# 3. Define target submodules (with name normalization)
# ==============================
# These are the actual submodule names inside each transformer layer
raw_submodule_names = [
    "self_attn.q_proj",
    "self_attn.k_proj",
    "self_attn.v_proj",
    "self_attn.o_proj",
    "mlp.gate_proj",
    "mlp.up_proj",
    "mlp.down_proj",
]

# Normalize names according to your rules:
# - gate_proj → up_proj
# - q_proj, v_proj → k_proj
def normalize_name(name):
    if name == "mlp.gate_proj":
        return "mlp.up_proj"
    elif name in ("self_attn.q_proj", "self_attn.v_proj"):
        return "self_attn.k_proj"
    else:
        return name

# Get unique normalized names
normalized_names = sorted(set(normalize_name(n) for n in raw_submodule_names))
print(f"Will collect activations for: {normalized_names}")

# ==============================
# 4. Hook class (per submodule per layer)
# ==============================
class SubmoduleHook:
    def __init__(self, layer_idx, submodule_name, save_base_dir):
        self.layer_idx = layer_idx
        self.submodule_name = submodule_name
        self.save_path = os.path.join(save_base_dir, submodule_name)
        os.makedirs(self.save_path, exist_ok=True)
        self.calib = None
        self.total_tokens = 0
        self.hook = None

    def hook_fn(self, module, input, output):
        with torch.no_grad():
            inp = input[0].detach()
            if inp.isnan().any() or inp.isinf().any():
                return
            if inp.dtype != torch.float32:
                inp = inp.float()
            # inp = torch.clamp(inp, -10.0, 10.0)

            B, S, D = inp.shape
            inp_flat = inp.view(-1, D)
            gram = (inp_flat.t() @ inp_flat).double()

            if self.calib is None:
                self.calib = gram.cpu()
            else:
                self.calib += gram.cpu()
            self.total_tokens += B * S

    def register(self, module):
        self.hook = module.register_forward_hook(self.hook_fn)

    def save_and_close(self):
        if self.calib is not None:
            avg_gram = (self.calib / self.total_tokens).float()
            file_path = os.path.join(self.save_path, f"{self.layer_idx}.pkl")
            with open(file_path, 'wb') as f:
                pickle.dump(avg_gram, f)
        if self.hook:
            self.hook.remove()

# ==============================
# 5. Register hooks on all layers and submodules
# ==============================
hooks = []  # list of all hooks

layers = model.model.language_model.layers
num_layers = len(layers)

for layer_idx, layer in enumerate(layers):
    for raw_name in raw_submodule_names:
        norm_name = normalize_name(raw_name)
        # Avoid duplicate hooks: only register once per (layer, norm_name)
        # We'll ensure uniqueness by checking if a hook for this (layer, norm_name) already exists
        # But since we loop raw names, we can skip redundant registration
        try:
            submodule = layer.get_submodule(raw_name)
        except AttributeError:
            print(f"Warning: {raw_name} not found in layer {layer_idx}")
            continue

        # Create hook only if we haven't registered one for this (layer, norm_name) yet
        # Simple way: check if any existing hook matches
        already_registered = any(
            h.layer_idx == layer_idx and h.submodule_name == norm_name for h in hooks
        )
        if not already_registered:
            hook = SubmoduleHook(layer_idx, norm_name, save_base_dir)
            hook.register(submodule)
            hooks.append(hook)

print(f"Registered {len(hooks)} hooks across {num_layers} layers.")

# ==============================
# 6. Run inference over all samples
# ==============================
print("Running inference to collect activations...")
model.eval()

for i in tqdm(range(len(train_raw_dataset))):
    try:
        messages = [
            {
                "role": "user",
                "content": [
                    {"type": "image", "image": train_raw_dataset[i]["image"]},
                    {"type": "text", "text": train_raw_dataset[i]["question_for_eval"]},
                ],
            },
            {
                "role": "assistant",
                "content": [{"type": "text", "text": train_raw_dataset[i]["answer"]}],
            },
        ]

        inputs = processor.apply_chat_template(
            messages,
            tokenize=True,
            add_generation_prompt=True,
            return_dict=True,
            return_tensors="pt"
        ).to(model.device)

        with torch.no_grad():
            _ = model(**inputs)

    except Exception as e:
        print(f"Skipping sample {i} due to error: {e}")
        continue

# ==============================
# 7. Save all and clean up
# ==============================
print("Saving activation statistics...")
for hook in hooks:
    hook.save_and_close()

print(f"✅ Calibration data saved to: {save_base_dir}")
print("Directory structure example:")
print(f"{save_base_dir}/self_attn.k_proj/0.pkl")
print(f"{save_base_dir}/mlp.up_proj/0.pkl")

Loading dataset...


`torch_dtype` is deprecated! Use `dtype` instead!


Loading model and processor...
Will collect activations for: ['mlp.down_proj', 'mlp.up_proj', 'self_attn.k_proj', 'self_attn.o_proj']
Registered 112 hooks across 28 layers.
Running inference to collect activations...


100%|██████████| 256/256 [24:32<00:00,  5.75s/it]


Saving activation statistics...
✅ Calibration data saved to: /share/b.mohammad/sparsesvd/Qwen34bvl/data/calib/MathVerse
Directory structure example:
/share/b.mohammad/sparsesvd/Qwen34bvl/data/calib/MathVerse/self_attn.k_proj/0.pkl
/share/b.mohammad/sparsesvd/Qwen34bvl/data/calib/MathVerse/mlp.up_proj/0.pkl


In [3]:
model.model

Qwen3VLModel(
  (visual): Qwen3VLVisionModel(
    (patch_embed): Qwen3VLVisionPatchEmbed(
      (proj): Conv3d(3, 1024, kernel_size=(2, 16, 16), stride=(2, 16, 16))
    )
    (pos_embed): Embedding(2304, 1024)
    (rotary_pos_emb): Qwen3VLVisionRotaryEmbedding()
    (blocks): ModuleList(
      (0-23): 24 x Qwen3VLVisionBlock(
        (norm1): LayerNorm((1024,), eps=1e-06, elementwise_affine=True)
        (norm2): LayerNorm((1024,), eps=1e-06, elementwise_affine=True)
        (attn): Qwen3VLVisionAttention(
          (qkv): Linear(in_features=1024, out_features=3072, bias=True)
          (proj): Linear(in_features=1024, out_features=1024, bias=True)
        )
        (mlp): Qwen3VLVisionMLP(
          (linear_fc1): Linear(in_features=1024, out_features=4096, bias=True)
          (linear_fc2): Linear(in_features=4096, out_features=1024, bias=True)
          (act_fn): GELUTanh()
        )
      )
    )
    (merger): Qwen3VLVisionPatchMerger(
      (norm): LayerNorm((1024,), eps=1e-06, ele

In [2]:
from transformers import AutoProcessor
p = AutoProcessor.from_pretrained("Qwen/Qwen3-VL-4B-Instruct")
p.save_pretrained("/share/a.ammar/swift_svd/Qwen34bvl_20_v1")

/opt/conda/lib/python3.11/site-packages/torch/cuda/__init__.py:63: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]


[]